# 02b - Global Identity Demo Mode

نوتبوك بحثي مستقل: يقرأ `local_tracks.csv` فقط ويكتب `Output/tables/global_tracks_demo.csv`. لا يغيّر `zone_events.csv` أو `global_tracks.csv` أو أي مرحلة من الـpipeline الأساسي. كل صف في الناتج يحمل `is_demo_mode=True`؛ النتائج التي تعتمد على المظهر فقط تقريبية ولا تصلح كهوية نهائية أو قياس دقة.

In [ ]:
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
import importlib.util
import json
import math
import re
import sys

import cv2
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
import torch
from torch.nn import functional as torch_functional

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
TABLES_DIR = PROJECT_ROOT / 'Output' / 'tables'
RAW_DIR = PROJECT_ROOT / 'Data' / 'raw'
CONFIG_DIR = PROJECT_ROOT / 'Data' / 'config'
LOCAL_TRACKS_PATH = TABLES_DIR / 'local_tracks.csv'
CALIBRATION_PATH = CONFIG_DIR / 'camera_calibration.generated.json'
REID_CONFIG_PATH = CONFIG_DIR / 'mtmc_reid_config.json'
DEMO_SIMILARITY_THRESHOLD = 0.75
DEMO_AMBIGUITY_MARGIN = 0.05

if not LOCAL_TRACKS_PATH.exists():
    raise FileNotFoundError('local_tracks.csv is missing. Run Notebook 01 first.')
local_tracks = pd.read_csv(LOCAL_TRACKS_PATH)
required_columns = {'camera_id', 'frame_index', 'timestamp_sec', 'local_track_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'foot_x', 'foot_y'}
missing_columns = sorted(required_columns.difference(local_tracks.columns))
if missing_columns:
    raise ValueError(f'local_tracks.csv is missing required columns: {missing_columns}')
if local_tracks.empty:
    raise ValueError('local_tracks.csv is empty. Run Notebook 01 with readable videos first.')
print(f'Loaded {len(local_tracks):,} local detections from {local_tracks.camera_id.nunique()} camera(s).')


In [ ]:
# Copied from Notebook/mtmc_reid.py: calibration validation and camera-local tracklet preparation.
def infer_store_id(camera_id):
    match = re.search(r'(place_\d+)', str(camera_id), flags=re.IGNORECASE)
    return match.group(1).lower() if match else 'default_store'

def make_tracklet_id(store_id, camera_id, local_track_id):
    return f'{store_id}::{camera_id}::{local_track_id}'

def validate_homography(entry, camera_id):
    matrix = np.asarray(entry.get('homography'), dtype=np.float64)
    if matrix.shape != (3, 3) or not np.isfinite(matrix).all() or np.linalg.matrix_rank(matrix) < 3:
        raise ValueError(f'Invalid 3x3 homography for {camera_id}.')
    identity = np.allclose(matrix, np.eye(3), atol=1e-9)
    if identity and not bool(entry.get('reference_camera', False)):
        raise ValueError(f'Identity homography is allowed only for an explicit reference camera: {camera_id}')
    return matrix, 'reference_camera' if identity else 'configured'

calibrations = json.loads(CALIBRATION_PATH.read_text(encoding='utf-8')) if CALIBRATION_PATH.exists() else {}
camera_modes = {}
for camera_id in sorted(local_tracks.camera_id.astype(str).unique()):
    entry = calibrations.get(camera_id)
    if entry is None:
        camera_modes[camera_id] = 'demo_appearance_only'
        print(f'{camera_id}: no calibration -> demo_appearance_only')
        continue
    try:
        _, mode = validate_homography(entry, camera_id)
        camera_modes[camera_id] = mode
        note = ' (reference pixels, not meters)' if mode == 'reference_camera' else ''
        print(f'{camera_id}: {mode}{note}')
    except ValueError as error:
        camera_modes[camera_id] = 'demo_appearance_only'
        print(f'{camera_id}: invalid calibration ({error}) -> demo_appearance_only')

work = local_tracks.copy()
work['store_id'] = work.camera_id.map(infer_store_id)
work['tracklet_id'] = [make_tracklet_id(store, camera, local) for store, camera, local in work[['store_id', 'camera_id', 'local_track_id']].itertuples(index=False, name=None)]
work['calibration_mode'] = work.camera_id.map(camera_modes)
work['floor_x'] = np.nan
work['floor_y'] = np.nan
for camera_id, camera_rows in work.groupby('camera_id', sort=True):
    if camera_modes[str(camera_id)] not in {'configured', 'reference_camera'}:
        continue
    matrix, _ = validate_homography(calibrations[str(camera_id)], str(camera_id))
    source_points = camera_rows[['foot_x', 'foot_y']].to_numpy(dtype=np.float32).reshape(-1, 1, 2)
    floor_points = cv2.perspectiveTransform(source_points, matrix).reshape(-1, 2)
    work.loc[camera_rows.index, ['floor_x', 'floor_y']] = floor_points
tracklets = (work.sort_values(['tracklet_id', 'timestamp_sec', 'frame_index']).groupby(['store_id', 'camera_id', 'local_track_id', 'tracklet_id', 'calibration_mode'], as_index=False).agg(start_sec=('timestamp_sec', 'min'), end_sec=('timestamp_sec', 'max'), mean_foot_x=('foot_x', 'mean'), mean_foot_y=('foot_y', 'mean'), mean_floor_x=('floor_x', 'mean'), mean_floor_y=('floor_y', 'mean'), sample_count=('frame_index', 'size')))
print(f'Prepared {len(tracklets):,} camera-local tracklets.')


In [ ]:
# Copied from Notebook/mtmc_reid.py: local OSNet-AIN loading and crop embedding extraction.
# The backbone itself is a vendored third-party model implementation; no analytics function is imported from another project file.
def load_osnet_model(weights_path, device_name='auto'):
    source_path = PROJECT_ROOT / 'vendor' / 'torchreid_osnet' / 'osnet_ain.py'
    if not source_path.exists():
        raise FileNotFoundError(f'Vendored OSNet source is missing: {source_path}')
    spec = importlib.util.spec_from_file_location('demo_osnet_ain', source_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Could not load OSNet source: {source_path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    device = torch.device('cuda' if device_name == 'auto' and torch.cuda.is_available() else device_name if device_name != 'auto' else 'cpu')
    model = module.osnet_ain_x1_0(num_classes=1000, pretrained=False)
    checkpoint_path = PROJECT_ROOT / weights_path
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'OSNet checkpoint is missing: {checkpoint_path}')
    try:
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    target_state = model.state_dict()
    compatible = {str(key).removeprefix('module.'): value for key, value in state.items() if str(key).removeprefix('module.') in target_state and target_state[str(key).removeprefix('module.')].shape == value.shape}
    if not compatible:
        raise RuntimeError('No compatible OSNet checkpoint weights were found.')
    model.load_state_dict(compatible, strict=False)
    return model.to(device).eval(), device

def resolve_camera_videos(camera_ids):
    by_stem = {}
    for path in RAW_DIR.rglob('*'):
        if path.is_file() and path.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'}:
            by_stem.setdefault(path.stem, []).append(path)
    resolved = {}
    for camera_id in sorted(set(map(str, camera_ids))):
        matches = by_stem.get(camera_id, [])
        if len(matches) != 1:
            raise RuntimeError(f'Expected one raw video for {camera_id}, found {matches}')
        resolved[camera_id] = matches[0]
    return resolved

def extract_tracklet_embeddings(rows, model, device, config):
    rows = rows.copy()
    rows['box_area'] = (rows.x2 - rows.x1) * (rows.y2 - rows.y1)
    rows = rows[(rows.confidence >= config['min_confidence']) & (rows.box_area >= config['min_box_area'])]
    selected = []
    for _, group in rows.sort_values(['tracklet_id', 'frame_index']).groupby('tracklet_id', sort=True):
        positions = np.linspace(0, len(group) - 1, min(config['samples_per_tracklet'], len(group)), dtype=int)
        selected.extend(group.iloc[np.unique(positions)].to_dict('records'))
    selected = pd.DataFrame(selected)
    if selected.empty:
        return {}, {'requested_crops': 0, 'valid_crops': 0, 'embedded_tracklets': 0}
    videos = resolve_camera_videos(selected.camera_id.unique())
    crops = {}
    valid_crops = 0
    for camera_id, camera_rows in selected.groupby('camera_id', sort=True):
        capture = cv2.VideoCapture(str(videos[str(camera_id)]))
        if not capture.isOpened():
            raise RuntimeError(f'Could not open raw video for {camera_id}')
        try:
            for row in camera_rows.sort_values('frame_index').itertuples(index=False):
                capture.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame_index))
                ok, frame = capture.read()
                if not ok:
                    continue
                height, width = frame.shape[:2]
                x1, y1 = max(0, int(math.floor(row.x1))), max(0, int(math.floor(row.y1)))
                x2, y2 = min(width, int(math.ceil(row.x2))), min(height, int(math.ceil(row.y2)))
                if x2 <= x1 or y2 <= y1:
                    continue
                crop = cv2.cvtColor(cv2.resize(frame[y1:y2, x1:x2], (128, 256)), cv2.COLOR_BGR2RGB)
                tensor = torch.from_numpy(crop).permute(2, 0, 1).float().div(255.0)
                tensor = (tensor - torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)) / torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
                crops.setdefault(str(row.tracklet_id), []).append(tensor)
                valid_crops += 1
        finally:
            capture.release()
    embeddings = {}
    with torch.inference_mode():
        for tracklet_id, tensors in crops.items():
            outputs = []
            for start in range(0, len(tensors), config['batch_size']):
                outputs.append(torch_functional.normalize(model(torch.stack(tensors[start:start + config['batch_size']]).to(device)), p=2, dim=1).cpu())
            embeddings[tracklet_id] = torch_functional.normalize(torch.cat(outputs).mean(dim=0), p=2, dim=0).numpy()
    return embeddings, {'requested_crops': int(len(selected)), 'valid_crops': valid_crops, 'embedded_tracklets': len(embeddings)}

reid_config = json.loads(REID_CONFIG_PATH.read_text(encoding='utf-8'))
embedding_config = reid_config['reid']
if tracklets.camera_id.nunique() < 2:
    embeddings, embedding_report = {}, {'requested_crops': 0, 'valid_crops': 0, 'embedded_tracklets': 0}
    print('Only one tracked camera is available; OSNet extraction is skipped because no cross-camera candidate can exist.')
else:
    osnet, osnet_device = load_osnet_model(embedding_config['weights_path'], embedding_config.get('device', 'auto'))
    embeddings, embedding_report = extract_tracklet_embeddings(work, osnet, osnet_device, embedding_config)
    print(f"OSNet embeddings: {embedding_report['embedded_tracklets']:,} tracklets from {embedding_report['valid_crops']:,} crops.")


In [ ]:
# Copied from Notebook/mtmc_reid.py: strict gates, Hungarian one-to-one assignment, and union-find compatibility checks.
@dataclass(frozen=True)
class AssociationConfig:
    max_time_gap_sec: float = 12.0
    max_floor_distance: float = 120.0
    max_appearance_distance: float = 0.45
    appearance_weight: float = 0.65
    spatial_weight: float = 0.25
    temporal_weight: float = 0.10
    max_assignment_score: float = 0.78
    assignment_window_sec: float = 5.0

strict_config = AssociationConfig(**{key: value for key, value in reid_config['association'].items() if key in AssociationConfig.__dataclass_fields__})

def cosine_similarity(left, right):
    return float(np.clip(np.dot(left, right) / (np.linalg.norm(left) * np.linalg.norm(right)), -1.0, 1.0))

def non_overlapping(left, right):
    return float(left.end_sec) < float(right.start_sec) or float(right.end_sec) < float(left.start_sec)

def make_candidates(tracklets, embeddings):
    candidates, ambiguous = [], []
    for left, right in combinations(tracklets.itertuples(index=False), 2):
        if left.store_id != right.store_id or left.camera_id == right.camera_id:
            continue
        left_strict = left.calibration_mode in {'configured', 'reference_camera'}
        right_strict = right.calibration_mode in {'configured', 'reference_camera'}
        if left.tracklet_id not in embeddings or right.tracklet_id not in embeddings:
            continue
        similarity = cosine_similarity(embeddings[left.tracklet_id], embeddings[right.tracklet_id])
        pair = {'store_id': left.store_id, 'camera_id_a': left.camera_id, 'camera_id_b': right.camera_id, 'tracklet_id_a': left.tracklet_id, 'tracklet_id_b': right.tracklet_id, 'association_time_sec': max(left.start_sec, right.start_sec), 'similarity': similarity}
        if left_strict and right_strict:
            time_gap = max(0.0, float(left.start_sec) - float(right.end_sec), float(right.start_sec) - float(left.end_sec))
            floor_distance = float(math.hypot(left.mean_floor_x - right.mean_floor_x, left.mean_floor_y - right.mean_floor_y))
            appearance_distance = 1.0 - similarity
            if time_gap > strict_config.max_time_gap_sec or floor_distance > strict_config.max_floor_distance or appearance_distance > strict_config.max_appearance_distance:
                continue
            score = strict_config.appearance_weight * (appearance_distance / strict_config.max_appearance_distance) + strict_config.spatial_weight * (floor_distance / strict_config.max_floor_distance) + strict_config.temporal_weight * (time_gap / strict_config.max_time_gap_sec)
            pair.update({'score': score, 'matching_mode': 'strict_calibrated'})
            if score <= strict_config.max_assignment_score:
                candidates.append(pair)
            elif score <= strict_config.max_assignment_score + DEMO_AMBIGUITY_MARGIN:
                ambiguous.append(pair)
        elif non_overlapping(left, right):
            pair.update({'score': 1.0 - similarity, 'matching_mode': 'demo_appearance_only'})
            if similarity >= DEMO_SIMILARITY_THRESHOLD:
                candidates.append(pair)
            elif similarity >= DEMO_SIMILARITY_THRESHOLD - DEMO_AMBIGUITY_MARGIN:
                ambiguous.append(pair)
    columns = ['store_id', 'camera_id_a', 'camera_id_b', 'tracklet_id_a', 'tracklet_id_b', 'association_time_sec', 'similarity', 'score', 'matching_mode']
    return pd.DataFrame(candidates, columns=columns), pd.DataFrame(ambiguous, columns=columns)

def hungarian_assign(candidates):
    if candidates.empty:
        return candidates.copy()
    rows = candidates.copy()
    rows['assignment_window'] = np.floor(rows.association_time_sec / strict_config.assignment_window_sec).astype(int)
    selected = []
    for _, group in rows.groupby(['store_id', 'camera_id_a', 'camera_id_b', 'assignment_window'], sort=True):
        left_ids, right_ids = sorted(group.tracklet_id_a.unique()), sorted(group.tracklet_id_b.unique())
        cost = np.full((len(left_ids) + len(right_ids), len(left_ids) + len(right_ids)), strict_config.max_assignment_score)
        left_index, right_index = {value: index for index, value in enumerate(left_ids)}, {value: index for index, value in enumerate(right_ids)}
        lookup = {}
        for index, row in group.iterrows():
            key = (left_index[row.tracklet_id_a], right_index[row.tracklet_id_b])
            if row.score < cost[key]:
                cost[key], lookup[key] = row.score, index
        assigned_rows, assigned_columns = linear_sum_assignment(cost)
        selected.extend(lookup[(row, column)] for row, column in zip(assigned_rows, assigned_columns) if (row, column) in lookup)
    return rows.loc[selected].reset_index(drop=True) if selected else rows.iloc[0:0].copy()

def build_global_mapping(tracklets, accepted):
    parent = {tracklet_id: tracklet_id for tracklet_id in tracklets.tracklet_id}
    def find(value):
        while parent[value] != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value
    for row in accepted.sort_values(['score', 'tracklet_id_a', 'tracklet_id_b']).itertuples(index=False):
        first, second = find(row.tracklet_id_a), find(row.tracklet_id_b)
        if first != second:
            parent[second] = first
    groups = {}
    for tracklet_id in parent:
        groups.setdefault(find(tracklet_id), []).append(tracklet_id)
    global_ids = {tracklet_id: f'demo_global_{index:06d}' for index, group in enumerate(sorted(groups.values(), key=lambda values: min(values)), start=1) for tracklet_id in group}
    return global_ids

candidates, ambiguous = make_candidates(tracklets, embeddings)
accepted_matches = hungarian_assign(candidates)
global_ids = build_global_mapping(tracklets, accepted_matches)
print(f'Candidate matches: {len(candidates):,}; accepted matches: {len(accepted_matches):,}; ambiguous near-threshold rejections: {len(ambiguous):,}.')


In [ ]:
mapping = tracklets[['store_id', 'camera_id', 'local_track_id', 'tracklet_id', 'calibration_mode']].copy()
mapping['global_track_id'] = mapping.tracklet_id.map(global_ids)
global_tracks_demo = work.merge(mapping, on=['store_id', 'camera_id', 'local_track_id', 'tracklet_id', 'calibration_mode'], how='left', validate='many_to_one')
global_tracks_demo['is_demo_mode'] = True
evidence_by_tracklet = {}
for row in accepted_matches.itertuples(index=False):
    evidence_by_tracklet[row.tracklet_id_a] = row.matching_mode
    evidence_by_tracklet[row.tracklet_id_b] = row.matching_mode
global_tracks_demo['matching_evidence_mode'] = global_tracks_demo.tracklet_id.map(evidence_by_tracklet).fillna('unmatched_camera_local_demo')
output_path = TABLES_DIR / 'global_tracks_demo.csv'
global_tracks_demo.to_csv(output_path, index=False)
print(f'Saved {len(global_tracks_demo):,} rows: {output_path.relative_to(PROJECT_ROOT)}')


In [ ]:
print(f'Global demo IDs: {mapping.global_track_id.nunique():,}')
print(f'Accepted matches: {len(accepted_matches):,}')
print(f'Ambiguous near-threshold candidates rejected: {len(ambiguous):,}')
print('WARNING: This artifact is a demo. Appearance-only associations are approximate and require a labeled evaluation sample before any real use.')
global_tracks_demo[['camera_id', 'local_track_id', 'global_track_id', 'calibration_mode', 'is_demo_mode']].head()
